In [ ]:
import torch
from torch_geometric.datasets import Planetoid

# Load the PubMed dataset
dataset = Planetoid(root='/tmp/PubMed', name='PubMed')


/Users/rojankarki/Projects/LearningML/GNN/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Processing...
Done!


In [7]:
data = dataset[0]
data

Data(x=[19717, 500], edge_index=[2, 88648], y=[19717], train_mask=[19717], val_mask=[19717], test_mask=[19717])

In [9]:
print(f'Dataset: {dataset}')
print(f"Nodes (papers):        {data.num_nodes}")
print(f"Edges (citations):     {data.num_edges}")
print(f'Number of Classes (topics): {dataset.num_classes}')
print(f"Node feature dim:      {dataset.num_node_features}  (TF-IDF vocab)")
print(f"Training nodes:        {int(data.train_mask.sum())}  "
      f"({100*int(data.train_mask.sum())/data.num_nodes:.1f}% of all nodes labeled)")
print(f"Validation nodes:      {int(data.val_mask.sum())}")
print(f"Test nodes:            {int(data.test_mask.sum())}")


Dataset: PubMed()
Nodes (papers):        19717
Edges (citations):     88648
Number of Classes (topics): 3
Node feature dim:      500  (TF-IDF vocab)
Training nodes:        60  (0.3% of all nodes labeled)
Validation nodes:      500
Test nodes:            1000


In [10]:
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        # Layer 1: message pass + aggregate + update, then non-linearity
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)

        # Layer 2: project down to class logits
        x = self.conv2(x, edge_index)
        return x


In [11]:
def train(model, optimizer):
    model.train()
    optimizer.zero_grad()
    out = model(data)
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()

@torch.no_grad()
def evaluate(model, mask):
    model.eval()
    out = model(data)
    pred = out.argmax(dim=1)
    correct = (pred[mask] == data.y[mask]).sum()
    return int(correct) / int(mask.sum())

In [12]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data = data.to(device)

In [13]:
epochs = 200
model = GCN(
    in_channels=dataset.num_node_features,
    hidden_channels=16, 
    out_channels=dataset.num_classes
    ).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
print("=== Training ===")
for epoch in range(epochs):
    loss = train(model, optimizer)
    val_acc = evaluate(model, data.val_mask)
    print(f"Epoch {epoch:3d} | Loss {loss:.4f} | Val Acc {val_acc:.4f}")


test_acc = evaluate(model, data.test_mask)
print(f"=== Final Test Accuracy: {test_acc:.4f} ===")

=== Training ===
Epoch   0 | Loss 1.0957 | Val Acc 0.3780
Epoch   1 | Loss 1.0787 | Val Acc 0.5000
Epoch   2 | Loss 1.0630 | Val Acc 0.6180
Epoch   3 | Loss 1.0387 | Val Acc 0.6540
Epoch   4 | Loss 1.0211 | Val Acc 0.6720
Epoch   5 | Loss 0.9948 | Val Acc 0.6860
Epoch   6 | Loss 0.9711 | Val Acc 0.6880
Epoch   7 | Loss 0.9489 | Val Acc 0.6920
Epoch   8 | Loss 0.9317 | Val Acc 0.7040
Epoch   9 | Loss 0.8872 | Val Acc 0.7080
Epoch  10 | Loss 0.8810 | Val Acc 0.7180
Epoch  11 | Loss 0.8689 | Val Acc 0.7160
Epoch  12 | Loss 0.8370 | Val Acc 0.7220
Epoch  13 | Loss 0.8216 | Val Acc 0.7160
Epoch  14 | Loss 0.8042 | Val Acc 0.7300
Epoch  15 | Loss 0.7439 | Val Acc 0.7300
Epoch  16 | Loss 0.7471 | Val Acc 0.7320
Epoch  17 | Loss 0.7214 | Val Acc 0.7360
Epoch  18 | Loss 0.7005 | Val Acc 0.7360
Epoch  19 | Loss 0.6652 | Val Acc 0.7380
Epoch  20 | Loss 0.6719 | Val Acc 0.7360
Epoch  21 | Loss 0.6108 | Val Acc 0.7360
Epoch  22 | Loss 0.6145 | Val Acc 0.7380
Epoch  23 | Loss 0.5973 | Val Acc 0.7420